# ML-03 - Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/8-BitRhyon/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

Lane 4 (provisional): CTR and engagement opportunity scoring for content review.
This notebook maps the lane onto the ML loop: task type, target, success metric, unit of
analysis, and the argument for why a learned ranking earns its place over a fixed rule.

All numbers come from the starter slice, `data/raw/content_refresh_anonymized.csv` (30,000 rows,
32 clients, trailing 90-day metrics). The forward-looking version of the same frame is built on
the warehouse in ML-04.

This notebook does not train a model. It decides what would be trained, on what, and how it
would be judged.


## 1. My lane as an ML task (type)

**Task type: ranking (scoring, then ordering).** Not classification, not clustering.

The actor does not ask "is this page bad?" one page at a time. They ask "which 140 pages do I
review first this month?", and the answer is an ordered queue. A classifier can produce the score
behind the order, but the deliverable and the metric are ranking artefacts.

### The four framing questions

| Question | Answer |
|---|---|
| What decision improves? | Which content pages a reviewer opens first next month, given capacity for 140 reviews over 7 weeks. |
| Who acts, and what do they do? | A content editor works down the queue and rewrites the title, meta description, or opening section of each page. |
| What does a wrong call cost? | A wasted review slot, and a missed page that keeps under-converting attention. Missing a high-impression page costs more than spending a slot on a small one. |
| Why data or ML? | The value of a review depends on CTR gap, position, volume, and ranking stability together. Section 5 shows that two plausible fixed rules share zero pages out of 140, so the choice of rule is the whole decision. |

### One-paragraph frame

> For a content editor deciding which pages to review first next month, we will build a ranked
> queue of page-months from the warehouse's daily search performance, scoring each page by its
> risk of under-converting clicks relative to its position tier, measured by precision@140 on
> clients not seen in training. A wrong call costs a review slot and a missed high-impression
> page. A plain rule is not enough because the signals interact and the obvious rule spends the
> whole budget on pages that carry little traffic. We will claim only observed and directional
> results.


In [1]:
REVIEW_BUDGET = 140   # pages, the capacity assumption carried from ML-02
REVIEW_WEEKS = 7

print(f"review budget: {REVIEW_BUDGET} pages over {REVIEW_WEEKS} weeks")
print(f"pace: {REVIEW_BUDGET / REVIEW_WEEKS:.0f} pages per week")
print(f"one review per working day is about {REVIEW_BUDGET / REVIEW_WEEKS / 5:.1f} reviews per day")


review budget: 140 pages over 7 weeks
pace: 20 pages per week
one review per working day is about 4.0 reviews per day


## 2. Target or proxy

**What I predict: the forward CTR of a page, relative to its position-tier peers.**

The real label, built on the warehouse in ML-04, is `low_ctr_next`: 1 when a page's April 2026
CTR is below the March 2026 CTR median of its own position tier, for pages with at least 100
April impressions. The outcome is measured in a window that has not happened at the decision
moment.

### Observed or defined?

The outcome is observed: clicks and impressions are counted in the later month. The threshold is
defined: a median split on March data. That is a deliberate choice with a cost. The model learns
to rank against that definition, not against revenue. Section 3's metric and Section 5's rule
baseline keep that honest, and if the definition moves, the queue moves.

### Why this is not the `is_declining_label` trap

`is_declining_label` is computed from `trend_direction`, which comes from `trend_pct` inside the
same 90-day snapshot. A model handed those columns learns the pipeline's rule, not the world.
This lane's label comes from a later window and never appears as a feature.

### The sketch on the starter data

The starter file is one 90-day snapshot, so it cannot produce a forward label. The sketch below
is cross-sectional: `low_ctr_vs_tier` = 1 when a page's 90-day CTR is below the 90-day CTR median
of its position tier. It shows the shape and base rate of the target. The forward version is
built on the warehouse next.


In [2]:
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd
import numpy as np

DATA_REL = Path("data/raw/content_refresh_anonymized.csv")
REPO_URL = "https://github.com/8-BitRhyon/flyrank-ml-internship"

# Colab runs from the repo root only if the notebook was opened from a clone.
if "google.colab" in sys.modules and not DATA_REL.exists():
    if not Path("flyrank-ml-internship").is_dir():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "flyrank-ml-internship"], check=True)
    os.chdir("flyrank-ml-internship")

# Locally, walk up from the notebook folder until the starter CSV is reachable.
while not DATA_REL.exists() and Path.cwd() != Path.cwd().parent:
    os.chdir("..")

df = pd.read_csv(DATA_REL)

# Lane slice: enough 90-day traffic for a stable ratio, and a real position value.
# 1,500 impressions over 90 days is about 500 per month, the same floor the warehouse contract uses.
slice_df = df[(df.impressions_90d >= 1500) & (df.avg_position > 0)].copy().reset_index(drop=True)

# Same four tiers as the warehouse contract, cut on the 90-day average position.
slice_df["tier"] = pd.cut(
    slice_df.avg_position, [0, 3, 10, 20, np.inf],
    labels=["top_3", "page_1", "page_2", "deep"],
).astype(str)

tier_median_ctr = slice_df.groupby("tier").ctr.median()
slice_df["low_ctr_vs_tier"] = (slice_df.ctr < slice_df.tier.map(tier_median_ctr)).astype(int)

print(f"starter rows: {len(df):,} across {df.client_id.nunique()} clients")
print(f"lane slice:   {len(slice_df):,} rows across {slice_df.client_id.nunique()} clients")
print(f"slice share:  {len(slice_df) / len(df):.1%} of the starter file")
print("\nmedian 90-day CTR by position tier (the sketch threshold):")
print(tier_median_ctr.round(3).to_string())
print(f"\nsketch label base rate: {slice_df.low_ctr_vs_tier.mean():.3f}")
print(slice_df.low_ctr_vs_tier.value_counts().rename({0: "not low", 1: "low"}).to_string())


starter rows: 30,000 across 32 clients
lane slice:   11,621 rows across 28 clients
slice share:  38.7% of the starter file

median 90-day CTR by position tier (the sketch threshold):
tier
deep      0.10
page_1    0.25
page_2    0.19
top_3     0.27

sketch label base rate: 0.491
low_ctr_vs_tier
not low    5920
low        5701


## 3. Success metric

**Metric: precision@140.**

Of the 140 pages the queue ranks first, the share that carry the forward label. That is the
number the editor experiences: of every slot spent, how many were worth it.

Why not the usual alternatives:

- **Accuracy** is dominated by the base rate and says nothing about the top of the queue.
- **ROC-AUC** measures ordering across all 11,621 pages, including the 99% the editor will never reach.
- **Recall@140** matters, but it is capped by the budget. With K/N near 1.2%, the queue can only ever reach a small share of the positive set. Report it, do not optimise it alone.

### The bar, set before training

- **precision@140 at least 0.80** on clients not seen in training
- at least **15 points above the rule baseline** at the same K

With a sketch base rate near 0.49, random selection scores about 0.49. A queue at 0.80 is roughly
1.6x random. If the model cannot clear the rule baseline, the rule ships.


In [3]:
K = 140
N = len(slice_df)
base_rate = slice_df.low_ctr_vs_tier.mean()
positives = int(slice_df.low_ctr_vs_tier.sum())

print(f"K = {K} reviews")
print(f"N = {N:,} pages in the working slice")
print(f"K / N = {K / N:.2%} of the slice can be reviewed")
print(f"sketch base rate = {base_rate:.3f}")
print(f"random precision@{K} = {base_rate:.3f}")
print(f"recall ceiling at this budget = {K / positives:.1%} of positives")
print(f"target precision@{K} = 0.80, a {0.80 / base_rate:.1f}x lift over random")


K = 140 reviews
N = 11,621 pages in the working slice
K / N = 1.20% of the slice can be reviewed
sketch base rate = 0.491
random precision@140 = 0.491
recall ceiling at this budget = 2.5% of positives
target precision@140 = 0.80, a 1.6x lift over random


## 4. The unit of analysis, as a real dataframe

**One row = one content page, over one trailing 90-day window.**

The starter file has no date column, so its row is a 90-day aggregate, not a page-month. The
production frame in ML-04 rolls the warehouse's daily rows up to one page-month. Same decision,
same fields, finer time grain.

The slice keeps pages with at least 1,500 impressions over 90 days and `avg_position > 0`.
Remember the data dictionary: `avg_position = 0` means "no position data", not rank zero.


In [4]:
unit_cols = [
    "content_id", "client_id", "impressions_90d", "clicks_90d",
    "ctr", "avg_position", "tier", "low_ctr_vs_tier",
]

assert slice_df.content_id.is_unique, "one row per content page"

print("one row = one content page over a trailing 90-day window")
print(f"rows: {len(slice_df):,} | clients: {slice_df.client_id.nunique()} | unique content ids: {slice_df.content_id.nunique():,}")
print(f"impressions covered: {int(slice_df.impressions_90d.sum()):,}")
print(f"clicks covered: {int(slice_df.clicks_90d.sum()):,}")
slice_df[unit_cols].head(8)


one row = one content page over a trailing 90-day window
rows: 11,621 | clients: 28 | unique content ids: 11,621
impressions covered: 149,758,826
clicks covered: 469,430


,content_id,client_id,impressions_90d,clicks_90d,ctr,avg_position,tier,low_ctr_vs_tier
0,content_304f48230142,client_f369cb89fc,3803,29,0.76,10.6,page_2,0
1,content_a1fb4e703a9e,client_4e07408562,15320,7,0.05,20.3,deep,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,0.09,36.5,deep,1
3,content_331d6c4de07b,client_19581e27de,11751,58,0.49,6.2,page_1,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,24,0.13,44.0,deep,0
5,content_d4084a4bc775,client_f369cb89fc,3970,1,0.03,8.5,page_1,1
6,content_a63219c6e95a,client_19581e27de,1724,1,0.06,21.2,deep,1
7,content_5e6c160719bc,client_6208ef0f77,32574,29,0.09,46.0,deep,1


## 5. Why ML beats a fixed rule here

The obvious fixed rule is "rank by lowest CTR". On this slice it chooses 140 pages with a total
expected click gap of about 662 clicks. It sorts on the ratio and ignores how much traffic sits
behind it, so the budget goes to pages whose total traffic is small.

A hand rule that weights the gap by volume, `(tier median CTR - page CTR) x impressions`, captures
about 21,300 clicks of gap with the same 140 slots. The two queues share **zero pages**.

So the rule choice is not a detail. It decides the entire queue. But the value rule is still
hand-built: the tier medians, the linear gap-times-volume trade-off, and the exclusion of position
stability and content age are all guesses.

The pattern is real but messy. Inside one tier, CTR runs from about 0.06% to 0.75% between the
10th and 90th percentile, at similar positions. A learned score can weigh gap, volume, position
stability, and age together instead of hard-coding one trade-off.

ML earns its place only if it beats the value rule at the same K on clients it has never seen.
If it does not, the rule ships. The rule is the baseline, not the strawman.


In [5]:
K = 140

# Value-weighted gap, used by rule B and to score both queues.
slice_df["click_gap"] = (
    slice_df.tier.map(tier_median_ctr) - slice_df.ctr
).clip(lower=0) * slice_df.impressions_90d / 100

# Rule A: the obvious one, lowest CTR first.
raw_queue = slice_df.nsmallest(K, "ctr")

# Rule B: value-weighted, gap against the tier median times impressions.
gap_queue = slice_df.nlargest(K, "click_gap")

overlap = len(set(raw_queue.content_id) & set(gap_queue.content_id))

summary = pd.DataFrame({
    "queue": ["lowest CTR (obvious rule)", "click gap x volume (value rule)"],
    "pages": [len(raw_queue), len(gap_queue)],
    "impressions_90d": [int(raw_queue.impressions_90d.sum()), int(gap_queue.impressions_90d.sum())],
    "clicks_90d_now": [int(raw_queue.clicks_90d.sum()), int(gap_queue.clicks_90d.sum())],
    "expected_click_gap": [round(raw_queue.click_gap.sum()), round(gap_queue.click_gap.sum())],
    "median_ctr_pct": [round(raw_queue.ctr.median(), 3), round(gap_queue.ctr.median(), 3)],
})

p10, p90 = slice_df[slice_df.tier == "page_1"].ctr.quantile([0.10, 0.90])
print(f"overlap between the two queues: {overlap} of {K} pages")
print(f"value rule captures {gap_queue.click_gap.sum() / raw_queue.click_gap.sum():.0f}x the expected gap of the obvious rule")
print(f"inside page_1, CTR runs {p10:.2f}% (p10) to {p90:.2f}% (p90), a {p90 / p10:.1f}x spread")
summary


overlap between the two queues: 0 of 140 pages
value rule captures 32x the expected gap of the obvious rule
inside page_1, CTR runs 0.06% (p10) to 0.75% (p90), a 12.5x spread


,queue,pages,impressions_90d,clicks_90d_now,expected_click_gap,median_ctr_pct
0,lowest CTR (obvious rule),140,443348,1,662,0.00
1,click gap x volume (value rule),140,15352238,13870,21300,0.07


## Self-check

- [x] Names the ML task type: ranking, with the reason it is not classification
- [x] Names the target or proxy: forward CTR against position-tier peers, plus the starter sketch
- [x] Names the success metric: precision@140, with a bar set before training
- [x] Shows the unit of analysis as a real dataframe: one row per content page over 90 days
- [x] Explains why this is an ML problem and not just a rule, with measured evidence
- [x] Ties the output to a real content action: an editor works down the ranked queue
- [x] The notebook ran top to bottom with no errors (executed before commit)
- [x] No client names, URLs, private queries, or tokens anywhere
- [x] Careful words throughout: observed, measured, directional, decision-support
- [x] Committed under `work/notebooks/`
